# **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already exists — pulling latest changes
Already up to date.


In [10]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

run_compile_all_cython: Found 11 Cython files in 5 folders...
run_compile_all_cython: All files will be compiled using your current python environment: '/usr/bin/python3'
Compiling [1/11]: MatrixFactorization_Cython_Epoch.pyx... 
/content/RecSys-Challenge-2025/CythonCompiler/compile_script.py:37: SyntaxWarning: invalid escape sequence '\.'
  extensionName = re.sub("\.pyx", "", fileToCompile)
In file included from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/ndarraytypes.h:1909,
                 from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/ndarrayobject.h:12,
                 from /usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/arrayobject.h:5,
                 from MatrixFactorization_Cython_Epoch.c:1252:
/usr/local/lib/python3.12/dist-packages/numpy/_core/include/numpy/npy_1_7_deprecated_api.h:17:2: warning: #warning "Using deprecated NumPy API, disable it with " "#define NPY_NO_DEPRECATED_API NPY_1_7_API_VERSION" []8;;

In [11]:
%%capture
if not IS_LOCAL:
    !pip install optuna

import optuna

In [12]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

Running on colab — storage at: /content/drive/MyDrive/RecSys


# **Load data**

In [13]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [14]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0

    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]

        if len(relevant_items)>0:
            num_eval+=1

            recommended_items = recommender.recommend(user_id, cutoff=at)

            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

# **Train a KNN with Jaccard similarity**

In [15]:
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender

SIMILARITY = "jaccard"

In [16]:
def perform_optimization(similarity, n_trials):
    # Define objective function for hyperparameter tuning
    STUDY_NAME = UserKNNCFRecommender.RECOMMENDER_NAME + "_" + similarity

    def objective_function(optuna_trial: optuna.trial.Trial) -> float:
        recommender_instance = UserKNNCFRecommender(URM_train)
        recommender_instance.fit(
            similarity=similarity,
            topK=optuna_trial.suggest_int("topK", 10, 1500),
            shrink=optuna_trial.suggest_int("shrink", 0, 2000),
            normalize=optuna_trial.suggest_categorical("normalize", [True, False]),
            feature_weighting=optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"])
        )

        return evaluate_recommender(recommender_instance, at=20)

    # Perform hyperparameter tuning
    save_results, optuna_study = hyperparameter_tuning(
        objective_function,
        study_name=STUDY_NAME,
        n_trials=n_trials
    )

    return save_results, optuna_study

## **Hyperparameter Tuning**

In [9]:
fd_results, optuna_study = perform_optimization(SIMILARITY, 100)

  0%|          | 0/100 [00:00<?, ?it/s]

Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 27095 (100.0%), 271.68 column/sec. Elapsed time 1.66 min
[I 2025-11-08 18:02:26,779] Trial 0 finished with value: 0.13166159391403198 and parameters: {'topK': 1280, 'shrink': 281, 'normalize': True, 'feature_weighting': 'BM25'}. Best is trial 0 with value: 0.13166159391403198.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 27095 (100.0%), 258.95 column/sec. Elapsed time 1.74 min
[I 2025-11-08 18:04:50,430] Trial 1 finished with value: 0.21146607398986816 and parameters: {'topK': 1486, 'shrink': 141, 'normalize': True, 'feature_weighting': 'TF-IDF'}. Best is trial 1 with value: 0.21146607398986816.
Unable to load Cython Compute_Similarity, reverting to Python
Similarity column 27095 (100.0%), 562.32 column/sec. Elapsed time 48.18 sec
[I 2025-11-08 18:05:56,132] Trial 2 finished with value: 0.11340254545211792 and parameters: {'topK': 152, 'shrink': 538, 'normalize': False, 'f

In [10]:
optuna.visualization.plot_optimization_history(optuna_study)

In [11]:
optuna.visualization.plot_param_importances(optuna_study)

In [12]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [17]:
STUDY_NAME = UserKNNCFRecommender.RECOMMENDER_NAME + "_tuning_" + SIMILARITY

"""
Best Value: 0.22113299369812012
Best Params: {'topK': 378, 'shrink': 0, 'normalize': True, 'feature_weighting': 'none'}
"""

def jaccard_tuning_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = UserKNNCFRecommender(URM_train)
    recommender_instance.fit(
        similarity=SIMILARITY,
        topK=optuna_trial.suggest_int("topK", 300, 450),
        shrink=optuna_trial.suggest_int("shrink", 0, 20),
        normalize=True,
        feature_weighting='none'
    )

    return evaluate_recommender(recommender_instance, at=20)

# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    jaccard_tuning_function,
    study_name=STUDY_NAME,
    n_trials=20
)

  0%|          | 0/20 [00:00<?, ?it/s]

Similarity column 27095 (100.0%), 721.89 column/sec. Elapsed time 37.53 sec
[I 2025-11-08 21:05:06,427] Trial 2 finished with value: 0.22111085057258606 and parameters: {'topK': 337, 'shrink': 7}. Best is trial 2 with value: 0.22111085057258606.
Similarity column 27095 (100.0%), 712.91 column/sec. Elapsed time 38.01 sec
[I 2025-11-08 21:06:05,432] Trial 3 finished with value: 0.2212367057800293 and parameters: {'topK': 313, 'shrink': 2}. Best is trial 3 with value: 0.2212367057800293.
Similarity column 27095 (100.0%), 712.49 column/sec. Elapsed time 38.03 sec
[I 2025-11-08 21:07:03,931] Trial 4 finished with value: 0.2197524756193161 and parameters: {'topK': 334, 'shrink': 18}. Best is trial 3 with value: 0.2212367057800293.
Similarity column 27095 (100.0%), 713.53 column/sec. Elapsed time 37.97 sec
[I 2025-11-08 21:08:04,915] Trial 5 finished with value: 0.22090712189674377 and parameters: {'topK': 359, 'shrink': 3}. Best is trial 3 with value: 0.2212367057800293.
Similarity column 27

# **Best Params**

- Best Value: 0.22147764265537262
- Best Params: {'topK': 301, 'shrink': 0, 'normalize': True, 'feature_weighting': 'none'}